# Week 6 — Encoder odometry: model versus real robot

The task and supplied Python, QRS and modeling notebook were reviewed. The provided navigation_data.csv is a different waypoint experiment with unknown provenance; it is not used as this student's digit run.

**Status:** graphical QRS tested in TRIK Studio; the simulator visibly completed 39 in 180.17 s. Its actual raw log is Husanboy_odometry_data_sim.txt. The real-robot log and tape-measured endpoint remain pending. No measurements or experimental conclusions are fabricated. The trajectory draws **39**. The QRS graphical program contains its own 14 target coordinates, turn-first control, marker lifting between digits, and encoder logging. Simulator calibration is wheel radius 0.028 m and track width 0.154 m; measure the hardware geometry before its run.

The program converts encoder degrees to metres, applies midpoint integration and uses positive counterclockwise heading: Δθ=(dR−dL)/B. The same controller is used in both environments. All motion, including the connector between digits, is logged.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Colab: upload both recorded files. Local: place them next to this notebook.
try:
    from google.colab import files
except ImportError:
    pass
else:
    uploaded = files.upload()

names = ['Husanboy_odometry_data_sim.txt', 'Husanboy_odometry_data_real.txt']
missing = [name for name in names if not Path(name).exists()]
print('Missing recorded logs: ' + ', '.join(missing) if missing else 'Both logs found.')

In [ ]:
def load_log(name):
    data = np.loadtxt(name, delimiter=',', ndmin=2)
    if data.shape[1] != 3 or len(data) < 2 or not np.isfinite(data).all():
        raise ValueError(name + ': need at least two finite x,y,time rows, no header')
    if np.any(np.diff(data[:,2]) <= 0):
        raise ValueError(name + ': timestamps must strictly increase')
    data[:,2] -= data[0,2]
    return data

if not missing:
    sim, real = [load_log(name) for name in names]
    plt.figure(figsize=(7,7))
    plt.plot(sim[:,0], sim[:,1], '--', label='TRIK simulator')
    plt.plot(real[:,0], real[:,1], label='Real robot odometry')
    plt.scatter([0], [0], c='black', label='Start')
    plt.axis('equal'); plt.grid(); plt.legend()
    plt.xlabel('x, m'); plt.ylabel('y, m')
    plt.title('Model versus real trajectory (including connecting motion)')
    plt.show()
else:
    print('Trajectory comparison pending the two actual runs.')

In [ ]:
if not missing:
    # Restrict interpolation to shared time: do not extrapolate a stopped run.
    common = real[:,2] <= sim[-1,2]
    comparison = real[common]
    if len(comparison) < 2:
        raise ValueError('Insufficient shared time interval')
    xs = np.interp(comparison[:,2], sim[:,2], sim[:,0])
    ys = np.interp(comparison[:,2], sim[:,2], sim[:,1])
    err = np.hypot(comparison[:,0]-xs, comparison[:,1]-ys)
    plt.figure(figsize=(8,4))
    plt.plot(comparison[:,2]/1000, err)
    plt.xlabel('Time, s'); plt.ylabel('Position difference, m'); plt.grid()
    plt.title('Simulator versus real odometry at matched times')
    plt.show()
    print(f'Final shared-time difference: {err[-1]:.4f} m')
    print(f'Maximum shared-time difference: {err.max():.4f} m')
    print(f'RMS shared-time difference: {np.sqrt(np.mean(err**2)):.4f} m')
    print(f'Final endpoint difference: {np.linalg.norm(real[-1,:2]-sim[-1,:2]):.4f} m')
    print(f'Durations: simulator {sim[-1,2]/1000:.2f} s, real {real[-1,2]/1000:.2f} s')
else:
    print('Error calculations pending recorded logs.')

## Independently measured endpoint
Mark the starting axle midpoint and initial forward direction. Use that point as (0,0), +x forward, +y left. After the robot stops, use a tape to measure the final axle midpoint in this same frame. Enter signed coordinates in metres below. The difference between two odometry traces is not an absolute localization error; only an independent measurement can reveal that.

In [ ]:
measured_x_m = None
measured_y_m = None

if not missing and measured_x_m is not None and measured_y_m is not None:
    measured = np.array([measured_x_m, measured_y_m], dtype=float)
    delta = real[-1,:2] - measured
    print('Measured endpoint (m):', measured)
    print('Odometry endpoint (m):', real[-1,:2])
    print('Odometry minus measured (m):', delta)
    print(f'Absolute endpoint localization error: {np.linalg.norm(delta):.4f} m')
else:
    print('Tape measurement and real log are required for endpoint validation.')

## Discussion (complete after recording; 5-sentence template)

The real trajectory first diverged noticeably from the simulator near **[time / segment]**, with a maximum matched-time difference of **[value] m**. The largest divergence occurred during **[straight motion / turns / connecting motion]**. This is consistent with **[wheel slip / incorrect radius or track width / unequal wheels]**, although timing differences also affect the matched-time plot. The tape-measured endpoint was **[x, y] m**, compared with the odometry endpoint **[x, y] m**, giving **[value] m** absolute endpoint error. Encoder-only odometry cannot know it is wrong because it measures wheel rotation rather than independent position, and early heading errors propagate into later position estimates.